In [43]:
#import necessary libraries
import pandas as pd

### CCF

In [44]:
#creating ccf dataframe and cleaning it for use in calculations
# make sure ccf.csv and ccf.xlsx for final version contain the same carriers and ccfs, later methodologody cleaning step!
df_ccf = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/CCF.csv')
ccf_clean = df_ccf[['Carrier', 'Carbon content factor']].copy()
ccf_clean['Carbon content factor'] = pd.to_numeric(
    ccf_clean['Carbon content factor'], 
    errors='coerce'
)
ccf_clean = ccf_clean.dropna(subset=['Carbon content factor'])
ccf_clean

,Carrier,Carbon content factor
1,Hard coal,0.716
2,Brown coal,0.328
3,Peat,0.282
4,Natural gas,15.300
5,Crude oil,0.846
6,Refined oil,0.856
7,Petroleum coke,0.863
8,Coke (from coal),0.823
9,Petrochemical feedstock,0.869
11,Primary biomass,0.500


### Fossil inflows

In [45]:
#creating fossil inflows dataframe and cleaning it for use in calculations
# If refering to Excel, use: pd.read_excel('[name].xlsx', sheet_name='XXX')
df_fossil_inflows = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/Fossil inflows.csv')
df_fossil_inflows


,Flow,Added info,Amount in dataset,Unit in dataset,Amount_tonnes,Carrier,Type,Source
0,"Crude oil, NGL, refinery feedstocks, additives...",Indigenous production,19057.03,thousand_tonnes,1.905703e+07,Crude oil,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
1,"Crude oil, NGL, refinery feedstocks, additives...",Imports,497079.54,thousand_tonnes,4.970795e+08,Crude oil,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
2,"Crude oil, NGL, refinery feedstocks, additives...",Exports,14406.38,thousand_tonnes,1.440638e+07,Crude oil,Outflow,https://ec.europa.eu/eurostat/databrowser/view...
3,"Crude oil, NGL, refinery feedstocks, additives...",Change in stock,4042.54,thousand_tonnes,4.042544e+06,Crude oil,Stock,https://ec.europa.eu/eurostat/databrowser/view...
4,Oil products,Imports,288398.08,thousand_tonnes,2.883981e+08,Refined oil,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
...,...,...,...,...,...,...,...,...
59,Natural gas,Indigenous production,1361966.08,Terajoule (gross calorific value - GCV),1.225769e+06,Gas,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
60,Natural gas,Imports,13916863.71,Terajoule (gross calorific value - GCV),1.252518e+07,Gas,Inflow,https://ec.europa.eu/eurostat/databrowser/view...
61,Natural gas,Exports,2491951.72,Terajoule (gross calorific value - GCV),2.242757e+06,Gas,Outflow,https://ec.europa.eu/eurostat/databrowser/view...
62,Natural gas,Change in stock,-91658.46,Terajoule (gross calorific value - GCV),-8.249261e+04,Gas,Stock,https://ec.europa.eu/eurostat/databrowser/view...


In [46]:
df_fossil_inflows_clean = df_fossil_inflows.drop(columns=['Amount in dataset', 'Unit in dataset', 'Source'])
df_fossil_inflows_clean.tail(30)

,Flow,Added info,Amount_tonnes,Carrier,Type
34,Coke oven coke,Indigenous production,2.784774e+07,Hard coal,Inflow
35,Coke oven coke,Imports,7.005156e+06,Hard coal,Inflow
36,Coke oven coke,Exports,9.004755e+06,Hard coal,Outflow
37,Coke oven coke,Change in stock,2.228127e+06,Hard coal,Stock
38,Patent fuel,Imports,8.779200e+04,Hard coal,Inflow
39,Patent fuel,Exports,3.504900e+04,Hard coal,Outflow
40,Patent fuel,Change in stock,1.180600e+04,Hard coal,Stock
41,Brown coal briquettes,Indigenous production,4.960417e+06,Hard coal,Inflow
42,Brown coal briquettes,Imports,4.244370e+05,Hard coal,Inflow
43,Brown coal briquettes,Exports,1.138137e+06,Hard coal,Outflow


In [47]:
# Map so "Gas" flows find "Natural gas" factors in ccf 
mapping = {
    'Gas': 'Natural gas', 
    'Crude oil': 'Crude oil', 
    'Hard coal': 'Hard coal', 
    'Brown coal': 'Brown coal', 
    'Refined oil': 'Refined oil',
    'Peat': 'Peat'
}
df_fossil_inflows_clean['CCF_Match'] = df_fossil_inflows_clean['Carrier'].map(mapping)

# Merge 'Carbon content factor' column to fossil data
df_carbon_fossil_inflow = df_fossil_inflows_clean.merge(ccf_clean, left_on='CCF_Match', right_on='Carrier', how='left', suffixes=('', '_ccf'))

df_carbon_fossil_inflow['Amount_tonnes'] = pd.to_numeric(df_carbon_fossil_inflow['Amount_tonnes'], errors='coerce')
df_carbon_fossil_inflow['Carbon content factor'] = pd.to_numeric(df_carbon_fossil_inflow['Carbon content factor'], errors='coerce')

# multiply ccf with amount tonnes to get tonnes of carbon
df_carbon_fossil_inflow['tonnes_Carbon'] = df_carbon_fossil_inflow['Amount_tonnes'] * df_carbon_fossil_inflow['Carbon content factor']


# if you want to save the result, uncomment the following line:
# df_carbon_fossil_inflow.to_csv('Calculated_Fossil_Carbon_Fixed.csv', index=False)
# or inspect the columns, check for missing values, etc.:
df_carbon_fossil_inflow.info()
df_carbon_fossil_inflow.tail(15)


<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Flow                   64 non-null     str    
 1   Added info             64 non-null     str    
 2   Amount_tonnes          64 non-null     float64
 3   Carrier                64 non-null     str    
 4   Type                   64 non-null     str    
 5   CCF_Match              64 non-null     str    
 6   Carrier_ccf            64 non-null     str    
 7   Carbon content factor  64 non-null     float64
 8   tonnes_Carbon          64 non-null     float64
dtypes: float64(3), str(6)
memory usage: 4.6 KB


,Flow,Added info,Amount_tonnes,Carrier,Type,CCF_Match,Carrier_ccf,Carbon content factor,tonnes_Carbon
49,Peat,Indigenous production,1.910664e+06,Peat,Inflow,Peat,Peat,0.282,5.388072e+05
50,Peat,Imports,7.972500e+04,Peat,Inflow,Peat,Peat,0.282,2.248245e+04
51,Peat,Exports,2.864900e+04,Peat,Outflow,Peat,Peat,0.282,8.079018e+03
52,Peat,Change in stock,1.568558e+06,Peat,Stock,Peat,Peat,0.282,4.423334e+05
53,Peat products,Indigenous production,3.688500e+04,Peat,Inflow,Peat,Peat,0.282,1.040157e+04
54,Peat products,Imports,3.600000e+04,Peat,Inflow,Peat,Peat,0.282,1.015200e+04
55,Peat products,Exports,5.565000e+03,Peat,Outflow,Peat,Peat,0.282,1.569330e+03
56,Peat products,Change in stock,1.290000e+03,Peat,Stock,Peat,Peat,0.282,3.637800e+02
57,Oil shale and oil sands,Indigenous production,1.339837e+07,Crude oil,Inflow,Crude oil,Crude oil,0.846,1.133502e+07
58,Oil shale and oil sands,Change in stock,-1.367238e+06,Crude oil,Stock,Crude oil,Crude oil,0.846,-1.156683e+06


In [48]:
# this is the non_calc version because the primary and secondary inflows are still in the same dataframe and can be confused with eachother
df_carbon_fossil_inflow.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Fossil_Carbon_Inflow_non_calc.csv', index=False)

In [49]:
df_carbon_fossil_inflow['Flow'].unique()

<StringArray>
['Crude oil, NGL, refinery feedstocks, additives and oxygenates and other hydrocarbons',
                                                                         'Oil products',
                                                                            'Hard coal',
                                                                           'Anthracite',
                                                                              'Lignite',
                                                                          'Coking coal',
                                                                'Other bituminous coal',
                                                                           'Brown coal',
                                                                  'Sub-bituminous coal',
                                                                       'Coke oven coke',
                                                                          'Patent fuel',
       

In [50]:
# Xeperate the primary fossil inflows into a seperate dataflow. 
# This eurostat dataflow contains both primary and secondary fossil inflows, this means it is counting the carbon inflows double when summing the total

primary_fuels = [
    "Crude oil, NGL, refinery feedstocks, additives and oxygenates and other hydrocarbons",
    "Natural gas",
    "Hard coal",
    "Brown coal",
    "Oil shale and oil sands",
    "Peat"
]

df_primary_fossil = df_carbon_fossil_inflow[df_carbon_fossil_inflow["Flow"].isin(primary_fuels)]



In [65]:
def fossil_supply_calc(group):

    production = group.loc[group["Added info"] == "Indigenous production", "tonnes_Carbon"].sum()
    imports = group.loc[group["Added info"] == "Imports", "tonnes_Carbon"].sum()
    exports = group.loc[group["Added info"] == "Exports", "tonnes_Carbon"].sum()
    stock = group.loc[group["Added info"] == "Change in stock", "tonnes_Carbon"].sum()
    bunkers = group.loc[group["Added info"] == "International maritime bunkers", "tonnes_Carbon"].sum()

    return production + imports - exports - bunkers - stock


fossil_supply_by_fuel = df_primary_fossil.groupby("Flow").apply(fossil_supply_calc)
total_fossil_supply = fossil_supply_by_fuel.sum()

print(fossil_supply_by_fuel, 'the total fossil supply is:', total_fossil_supply)

Flow
Brown coal                                                                              7.422506e+07
Crude oil, NGL, refinery feedstocks, additives and oxygenates and other hydrocarbons    4.210437e+08
Hard coal                                                                               9.691252e+07
Natural gas                                                                             1.770591e+08
Oil shale and oil sands                                                                 1.249171e+07
Peat                                                                                    1.108773e+05
dtype: float64 the total fossil supply is: 781842995.8202001


In [52]:
fossil_supply_df = fossil_supply_by_fuel.reset_index()
fossil_supply_df.columns = ["Flow", "Fossil_supply_tC"]

fossil_supply_df
#fossil_supply_df["Type"] = "Fossil inflow"
fossil_supply_df.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Aggregated_Fossil_Carbon_Inflow.csv', index=False)

In [53]:
df_final_fossil_inflow = df_primary_fossil.drop(columns=['Amount_tonnes', 'CCF_Match', 'Carrier_ccf', 'Carbon content factor'])
df_final_fossil_inflow.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Calculated_Primary_Fossil_Carbon_Inflow.csv', index=False)

### GHG

In [54]:
#creating ghg outflows dataframe and cleaning it for use in calculations
# If refering to Excel, use: pd.read_excel('[name].xlsx', sheet_name='XXX')
df_ghg_outflows = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/_PC_ghg_emissions.csv')
df_ghg_outflows.columns


Index(['Flow', 'Added info', 'Unit in dataset', '#CO2 in dataset',
       'Amount_tonnes_CO2', '#CH4 in dataset', 'Amount_tonnes_CH4',
       '#CO in dataset', 'Amount_tonnes_CO', 'Sankey node flow name', 'Type',
       'Source'],
      dtype='str')

In [55]:
df_ghg_outflows_clean = df_ghg_outflows.drop(columns=['Unit in dataset', '#CO2 in dataset', '#CH4 in dataset', '#CO in dataset', 'Source'])
df_ghg_outflows_clean.info()
df_ghg_outflows_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Flow                   11 non-null     str    
 1   Added info             27 non-null     str    
 2   Amount_tonnes_CO2      31 non-null     float64
 3   Amount_tonnes_CH4      31 non-null     float64
 4   Amount_tonnes_CO       31 non-null     float64
 5   Sankey node flow name  31 non-null     str    
 6   Type                   31 non-null     str    
dtypes: float64(3), str(4)
memory usage: 4.1 KB


,Flow,Added info,Amount_tonnes_CO2,Amount_tonnes_CH4,Amount_tonnes_CO,Sankey node flow name,Type
0,1.A.1. Energy industries,1.A.1.a. Public electricity and heat production,5.679841e+08,119356.697800,283681.48470,Energy production,Outflow
1,NaN,1.A.1.b. Petroleum refining,9.290136e+07,2359.067881,17549.60913,Energy,Outflow
2,NaN,1.A.1.c. Manufacture of solid fuels and other ...,2.765303e+07,6357.444519,82132.55458,Energy,Outflow
3,1.A.2. Manufacturing industries and construction,1.A.2.a. Iron and steel,6.759795e+07,4597.164908,613217.15560,Manufacturing,Outflow
4,NaN,1.A.2.b. Non-ferrous metals,7.721146e+06,391.923416,12262.71470,Manufacturing,Outflow


In [56]:
# dropping ghost rows from the old excel file (all the 31 nan rows)

df_ghg_outflows_clean = df_ghg_outflows_clean.dropna(subset=['Sankey node flow name'])

df_ghg_outflows_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Flow                   11 non-null     str    
 1   Added info             27 non-null     str    
 2   Amount_tonnes_CO2      31 non-null     float64
 3   Amount_tonnes_CH4      31 non-null     float64
 4   Amount_tonnes_CO       31 non-null     float64
 5   Sankey node flow name  31 non-null     str    
 6   Type                   31 non-null     str    
dtypes: float64(3), str(4)
memory usage: 1.8 KB


In [57]:
ccf_library = df_ccf.dropna(subset=['Carbon content factor'])
ccf_library = dict(zip(ccf_library['Carrier'], ccf_library['Carbon content factor']))

# Print it to see your "Library"
print("Your Automated CCF Library:")
for carrier, value in ccf_library.items():
    print(f" - {carrier}: {value} kg C/kg")

Your Automated CCF Library:
 - Hard coal: 0.716 kg C/kg
 - Brown coal: 0.328 kg C/kg
 - Peat: 0.282 kg C/kg
 - Natural gas: 15.3 kg C/kg
 - Crude oil: 0.846 kg C/kg
 - Refined oil: 0.856 kg C/kg
 - Petroleum coke: 0.863 kg C/kg
 - Coke (from coal): 0.823 kg C/kg
 - Petrochemical feedstock: 0.869 kg C/kg
 - Primary biomass: 0.5 kg C/kg
 - Wood pellets / Timber of Biomass forestry: 0.476 kg C/kg
 - Crops & livestock: 0.48 kg C/kg
 - Non-food biomass: 0.62 kg C/kg
 - Limestone: 0.12 kg C/kg
 - Virgin plastics: 0.741 kg C/kg
 - Transport fuels: 0.856 kg C/kg
 - CO2 : 0.27 kg C/kg
 - CH4 : 0.75 kg C/kg
 - CO: 0.428 kg C/kg
 - Plastic waste: 0.741 kg C/kg
 - Plastics in buildings: 0.741 kg C/kg
 - Wood in construction: 0.476 kg C/kg
 - Bitumen in roads: 0.884 kg C/kg


In [58]:
CCF_FACTORS = {
    'Amount_tonnes_CO2': 0.27,
    'Amount_tonnes_CH4': 0.75,
    'Amount_tonnes_CO': 0.4286
}
		

# 3. Clean and Multiply in one loop
# This ensures columns are numeric and then creates new 'Carbon' columns
for gas, factor in CCF_FACTORS.items():
    if gas in df_ghg_outflows_clean.columns:
        df_ghg_outflows_clean[gas] = pd.to_numeric(df_ghg_outflows_clean[gas], errors='coerce').fillna(0)
        new_col_name = f'Carbon_from_{gas}'
        df_ghg_outflows_clean[new_col_name] = df_ghg_outflows_clean[gas] * factor

# 4. Total Carbon from all GHGs
# This sums up the new columns to give you the total carbon outflow
carbon_cols = [f'Carbon_from_{gas}' for gas in CCF_FACTORS.keys() if gas in df_ghg_outflows_clean.columns]
df_ghg_outflows_clean['Total_Carbon_GHG'] = df_ghg_outflows_clean[carbon_cols].sum(axis=1)

df_ghg_outflows_clean.head()

# Save result
#df_ghg_outflows_clean.to_csv('Calculated_GHG_Carbon.csv', index=False)

,Flow,Added info,Amount_tonnes_CO2,Amount_tonnes_CH4,Amount_tonnes_CO,Sankey node flow name,Type,Carbon_from_Amount_tonnes_CO2,Carbon_from_Amount_tonnes_CH4,Carbon_from_Amount_tonnes_CO,Total_Carbon_GHG
0,1.A.1. Energy industries,1.A.1.a. Public electricity and heat production,5.679841e+08,119356.697800,283681.48470,Energy production,Outflow,1.533557e+08,89517.523350,121585.884342,1.535668e+08
1,NaN,1.A.1.b. Petroleum refining,9.290136e+07,2359.067881,17549.60913,Energy,Outflow,2.508337e+07,1769.300911,7521.762473,2.509266e+07
2,NaN,1.A.1.c. Manufacture of solid fuels and other ...,2.765303e+07,6357.444519,82132.55458,Energy,Outflow,7.466318e+06,4768.083389,35202.012893,7.506288e+06
3,1.A.2. Manufacturing industries and construction,1.A.2.a. Iron and steel,6.759795e+07,4597.164908,613217.15560,Manufacturing,Outflow,1.825145e+07,3447.873681,262824.872890,1.851772e+07
4,NaN,1.A.2.b. Non-ferrous metals,7.721146e+06,391.923416,12262.71470,Manufacturing,Outflow,2.084709e+06,293.942562,5255.799520,2.090259e+06


In [59]:
df_ghg_outflows_clean.drop(columns=['Amount_tonnes_CO2', 'Amount_tonnes_CH4', 'Amount_tonnes_CO'], inplace=True)
df_ghg_outflows_clean = df_ghg_outflows_clean.rename(columns={
    'Carbon_from_Amount_tonnes_CO2': 'Carbon_from_CO2',
    'Carbon_from_Amount_tonnes_CH4': 'Carbon_from_CH4',
    'Carbon_from_Amount_tonnes_CO': 'Carbon_from_CO'
    })

In [60]:
df_ghg_outflows_clean.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Calculated_GHG_Carbon_Outflow.csv', index=False)